# Eat Safe, Love

## Notebook Set Up

In [ ]:
# Import dependencies
from pymongo import MongoClient
from pprint import pprint

In [ ]:
# Create an instance of MongoClient
mongo = MongoClient(port=27017)

In [ ]:
# assign the uk_food database to a variable name
db = mongo['uk_food']

In [ ]:
# review the collections in our database
db.list_collection_names()

In [ ]:
# assign the collection to a variable
establishments = db['establishments']

## Part 3: Exploratory Analysis
Unless otherwise stated, for each question: 
* Use `count_documents` to display the number of documents contained in the result.
* Display the first document in the results using `pprint`.
* Convert the result to a Pandas DataFrame, print the number of rows in the DataFrame, and display the first 10 rows.

### 1. Which establishments have a hygiene score equal to 20?

In [ ]:
# Find the establishments with a hygiene score of 20
query = {'scores.Hygiene': 20}
# Use count_documents to display the number of documents in the result
establishments.count_documents(query)
# Display the first document in the results using pprint
pprint(establishments.find_one(query))

In [ ]:
# Convert the result to a Pandas DataFrame
import pandas as pd
# Display the number of rows in the DataFrame
establishments_df = pd.DataFrame(list(establishments.find()))
# Display the first 10 rows of the DataFrame
establishments_df.head(10)

### 2. Which establishments in London have a `RatingValue` greater than or equal to 4?

In [ ]:
# Find the establishments with London as the Local Authority and has a RatingValue greater than or equal to 4.
query2 = {'LocalAuthorityName': {'$regex': 'London'},
          'RatingValue': {'$gte': 4}}

# Use count_documents to display the number of documents in the result
count2 = establishments.count_documents(query2)
print(count2)
# Display the first document in the results using pprint
pprint(establishments.find_one(query2))




In [ ]:
# Convert the result to a Pandas DataFrame
establishments_df2 = pd.DataFrame(establishments.find(query2))
# Display the number of rows in the DataFrame
len(establishments_df2)
# Display the first 10 rows of the DataFrame
establishments_df2.head(10)

### 3. What are the top 5 establishments with a `RatingValue` rating value of 5, sorted by lowest hygiene score, nearest to the new restaurant added, "Penang Flavours"?

In [ ]:
# Search within 0.01 degree on either side of the latitude and longitude.
# Rating value must equal 5
# Sort by hygiene score

degree_search = 0.01
latitude = 51.490142
longitude = 0.08384

query3 = {'RatingValue': 5, 
         'geocode.latitude': {'$lte': 51.500142, '$gte': 51.480142},
         'geocode.longitude': {'$lte': 0.09384, '$gte': 0.07384}}
sort = [('scores.Hygiene', 1)]



# Print the results
for result in db.establishments.find(query3).sort(sort).limit(5):
    print(result)


In [ ]:
# Convert result to Pandas DataFrame
import pandas as pd
# Display the number of rows in the DataFrame
establishments_df3 = pd.DataFrame(list(establishments.find()))
# Display the first 10 rows of the DataFrame
establishments_df3.head(10)

### 4. How many establishments in each Local Authority area have a hygiene score of 0?

In [ ]:
# Create a pipeline that: 
# 1. Matches establishments with a hygiene score of 0
match = {'$match': {'scores.Hygiene': 0}}
# 2. Groups the matches by Local Authority
group = {'$group': {'_id': '$LocalAuthorityName', 'count': {'$sum': 1}}}
# 3. Sorts the matches from highest to lowest
sort = {'$sort': {'count': -1}}
# Print the number of documents in the result
pipeline = [match, group, sort]
result = list(establishments.aggregate(pipeline))
print(len(result))
# Print the first 10 results
pprint(result[:10])

In [ ]:
# Convert the result to a Pandas DataFrame
establishments_df4 = pd.DataFrame(list(establishments.aggregate(pipeline)))
# Display the number of rows in the DataFrame
print(len(establishments_df4))
# Display the first 10 rows of the DataFrame
establishments_df4.head(10)